# Nemotron-Labs Unified ASR 0.6B — 25-language transcription on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Nemotron-Labs-Unified-ASR-0.6B-multilingual.ipynb)

[`nvidia/nemotron-labs-unified-asr-0.6b`](https://huggingface.co/nvidia/nemotron-labs-unified-asr-0.6b) is NVIDIA's **~0.6B** FastConformer-RNNT speech recognizer, published on Hugging Face on 27 October 2026. It is the multilingual version of `parakeet-unified-en-0.6b`: **one checkpoint for offline and streaming** ASR (streaming latency from 2080 ms down to 160 ms) in **25 European languages**, with punctuation and capitalization built in.

Why this model: the target language is passed as a **language-ID prompt** to the encoder (`target_lang="de"`), and there is an automatic mode (`target_lang="auto"`) when you do not know the language. The card reports 1.38 % WER on LibriSpeech test-clean (offline) and says it beats `parakeet-tdt-0.6b-v3` offline.

**Licence:** [OpenMDW-1.1](https://github.com/OpenMDW/OpenMDW/blob/main/1.1/LICENSE.OpenMDW-1.1). Commercial and non-commercial use are allowed. Keep the licence text and notices if you redistribute the weights.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has 16 GB of VRAM. The `.nemo` checkpoint is about **2.5 GB** (float32). Offline transcription of short clips should need roughly 3–4 GB of VRAM. That is an estimate from the checkpoint size plus short clips; it is not a measured Colab number yet.

**Why NeMo from GitHub:** the `target_lang` prompt path for this model needs a fix that is on NeMo `main` but not in the `nemo_toolkit==3.0.0` PyPI release. With 3.0.0, `transcribe(..., target_lang=...)` fails with `Unknown prompt key: 'None'`. The install cell pins NeMo commit `1c1b25339`.

**Not in this notebook:** chunked streaming inference (the card's `speech_to_text_streaming_infer_rnnt.py` script and the `buffered_rnnt.yaml` pipeline) and the ASR + LLM speech-translation cascade.

**Links:** [model card](https://huggingface.co/nvidia/nemotron-labs-unified-asr-0.6b) · [NeMo](https://github.com/NVIDIA-NeMo/NeMo)

In [ ]:
%pip install -q "nemo_toolkit[asr] @ git+https://github.com/NVIDIA-NeMo/NeMo.git@1c1b25339ca2b31794ba4b299a5ea6a2c83c64b5"

# NeMo main pinned to one commit (reports 3.1.0+1c1b25339). PyPI 3.0.0 does not pass
# target_lang into the Lhotse prompt dataset, so every transcribe() call fails.
# Colab's preinstalled torch / torchaudio are used as-is.

> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> NeMo upgrades a few packages Colab has already imported. They only take effect after a restart.

In [ ]:
import torch
import nemo

print("nemo", nemo.__version__, "| torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

major, minor = torch.cuda.get_device_capability()
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
free, total = torch.cuda.mem_get_info()
print(f"VRAM free: {free / 1024**3:.1f} GB of {total / 1024**3:.1f} GB")


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"VRAM allocated: {allocated:.2f} GB | peak: {peak:.2f} GB{suffix}")

## 1. Load the model

`ASRModel.from_pretrained` downloads the 2.5 GB `.nemo` file and restores `EncDecRNNTBPEModelWithPrompt`. The model stays in float32 (the card's default); at 0.6B that is small for a T4.

In [ ]:
import nemo.collections.asr as nemo_asr

MODEL_ID = "nvidia/nemotron-labs-unified-asr-0.6b"

torch.cuda.reset_peak_memory_stats()
asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name=MODEL_ID, map_location="cuda").eval()

n_params = sum(p.numel() for p in asr_model.parameters())
print("class:", type(asr_model).__name__)
print(f"parameters: {n_params / 1e6:.0f}M")
prompt_keys = sorted(asr_model.cfg.model_defaults.prompt_dictionary.keys())
print(f"{len(prompt_keys)} prompt keys, e.g.:", [k for k in prompt_keys if len(k) == 2][:30])
print_vram("after load")

## 2. Sample audio

* English: `sample1.flac` from the Hugging Face speech samples CDN (LibriSpeech-style read speech).
* German, French, Spanish: short phone-banking requests from [`PolyAI/minds14`](https://huggingface.co/datasets/PolyAI/minds14) (CC BY 4.0), fetched through the Hugging Face dataset viewer API with their reference transcripts. These are 8 kHz phone recordings; NeMo resamples them to 16 kHz.

In [ ]:
import requests
from IPython.display import Audio, display

UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}


def download(url, path):
    response = requests.get(url, headers=UA, timeout=120)
    response.raise_for_status()
    with open(path, "wb") as f:
        f.write(response.content)
    return path


def minds14_clip(config, index):
    rows = requests.get(
        "https://datasets-server.huggingface.co/rows",
        params={"dataset": "PolyAI/minds14", "config": config, "split": "train", "offset": index, "length": 1},
        headers=UA,
        timeout=120,
    ).json()["rows"]
    row = rows[0]["row"]
    path = download(row["audio"][0]["src"], f"minds14-{config}-{index}.wav")
    return path, row["transcription"]


clips = [("en", download("https://cdn-media.huggingface.co/speech_samples/sample1.flac", "en-sample1.flac"), None)]
for config, lang in [("de-DE", "de"), ("fr-FR", "fr"), ("es-ES", "es")]:
    path, reference = minds14_clip(config, 1)
    clips.append((lang, path, reference))

for lang, path, reference in clips:
    print(lang, path, "| reference:", reference)
    display(Audio(path))

## 3. Transcribe with an explicit language prompt

`transcribe(paths, target_lang=...)` is the card's offline call. Short ISO codes (`en`, `de`, `fr`, `es`) and locale codes (`de-DE`) are both in the prompt dictionary. Output has punctuation and capitalization; the minds14 references do not, so compare words, not formatting.

In [ ]:
import time

for lang, path, reference in clips:
    start = time.time()
    hyp = asr_model.transcribe([path], target_lang=lang, verbose=False)[0]
    print(f"[{lang}] {time.time() - start:.2f}s  {hyp.text}")
    if reference:
        print(f"      ref: {reference}")
print_vram("after explicit-language pass")

## 4. Automatic language mode

`target_lang="auto"` uses the language-agnostic prompt. The card recommends an explicit language for low-latency **streaming**; for offline clips like these, auto usually matches.

In [ ]:
paths = [path for _, path, _ in clips]
start = time.time()
auto_hyps = asr_model.transcribe(paths, target_lang="auto", batch_size=4, verbose=False)
print(f"batch of {len(paths)} clips in {time.time() - start:.2f}s\n")
for (lang, _, _), hyp in zip(clips, auto_hyps):
    print(f"[auto, true={lang}] {hyp.text}")
print_vram("after auto batch")

## 5. Word timestamps

`timestamps=True` returns word and segment offsets in seconds (8× subsampling → 80 ms steps). Handy for subtitles or for aligning with a diarizer like the [Nemotron-3 Diarization notebook](Nemotron-3-Diarization.ipynb).

In [ ]:
hyp = asr_model.transcribe([clips[0][1]], target_lang="en", timestamps=True, verbose=False)[0]
for w in hyp.timestamp["word"][:12]:
    print(f"{w['start']:6.2f}s – {w['end']:6.2f}s  {w['word']}")
print("...")
for seg in hyp.timestamp.get("segment", [])[:3]:
    print(f"segment {seg['start']:.2f}s – {seg['end']:.2f}s: {seg['segment']}")

## 6. Caveats and cleanup

**What this notebook is for.** Offline multilingual transcription on a free T4 with the card's NeMo API: explicit language prompt, auto mode, batching, and word timestamps.

**What it is not.** Streaming. The unified model streams with a *buffered* (not cache-aware) pipeline: the card's chunked RNNT script or `PipelineBuilder` with `buffered_rnnt.yaml` (left 5.6 s / chunk 0.56 s / right 0.56 s for 1.12 s latency). The left context is recomputed every chunk, so it is heavier than NVIDIA's cache-aware streaming models. Speech translation needs a second LLM and is also out of scope.

**Languages.** bg, hr, cs, da, nl, en, et, fi, fr, de, el, hu, it, lv, lt, mt, pl, pt, ro, sk, sl, es, sv, ru, uk. The prompt dictionary has more keys (shared with other NVIDIA prompt models), but only these 25 are trained for this checkpoint.

**T4 notes.** float32 weights, no bfloat16 needed. Keep single clips to a few minutes; for long recordings, split first.

**Related notebooks:** [Granite Speech 5.0 2B](granite-speech-5.0-2b-asr-translation.ipynb) (ASR + translation), [oruk-loop-153M](oruk-loop-153M-looped-asr.ipynb) (English ASR), [Whisper](AIBackends-Whisper-audio-transcription.ipynb), [Nemotron-3 Diarization](Nemotron-3-Diarization.ipynb).

Free the GPU before you open another notebook:

In [ ]:
import gc

print_vram("before cleanup")
del asr_model
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")